# Brussels Traffic Data Processing Pipeline

In [1]:
%reset -f

In [2]:
# Standard imports
import sys
sys.path.insert(0, '/home/ubuntu/prem')

import pandas as pd
import numpy as np
import gc
from tqdm import tqdm
import geopandas as gpd
from shapely import wkt

In [3]:
# Modular imports
from utils import log_memory, log_df_memory, load_data, save_detection_results
from filters.preprocessing import (
    filter_by_lifetime,
    attach_zones_to_objects,
    apply_footpath_zone_filter,
    compute_polygon_orientation,
    filter_parallel_vehicles,
    filter_static_objects
)
from regions.brussels.zones import get_lane_zones, get_footpath_zones, get_crosswalk_zones
from ssm.utils import load_config, assign_zones_to_vehicles
from ssm.m_drac import ModifiedDRAC
from ssm.spf import SafetyPotentialField

[SSM] Numba enabled with 7 threads


In [4]:
# Configuration
START_DATE = "2025-06-05"
END_DATE = "2025-06-05"
DATA_DIR = "/home/ubuntu/data/uploads/objects/clean"
OUTPUT_DIR = "/home/ubuntu/prem/results"

config = load_config("/home/ubuntu/prem/config.yaml")

print("="*70)
print("BRUSSELS TRAFFIC ANALYSIS")
print("="*70)
print(f"Date: {START_DATE} to {END_DATE}")
print("="*70)

BRUSSELS TRAFFIC ANALYSIS
Date: 2025-06-05 to 2025-06-05


## 1. Data Loading

In [5]:
print("\nLoading data...")
log_memory("Before loading")

df = load_data(DATA_DIR, START_DATE, END_DATE, dtypes=config['data']['dtypes'])

log_df_memory(df, "Loaded data")
print(f"Loaded {len(df):,} records")
df.reset_index(drop=True, inplace=True)


Loading data...
[MEMORY] Before loading: 197.9 MB


Loading data: 100%|██████████| 336/336 [00:04<00:00, 69.63it/s]


[DF MEMORY] Loaded data: 2941.0 MB (50,554,288 rows)
Loaded 50,554,288 records


## 2. Lifetime Filtering

In [6]:
print("\n" + "="*70)
print("Lifetime Filtering")
print("="*70)

df = filter_by_lifetime(df, config['preprocessing']['lifetime_filter']['min_lifespan'])
log_df_memory(df, "After lifetime filter")


Lifetime Filtering
[lifespan filter] Removed 6,767 short-lived IDs
  Before: 93,251 IDs (50,554,288 rows)
  After: 86,484 IDs (50,321,862 rows)
[DF MEMORY] After lifetime filter: 3311.4 MB (50,321,862 rows)


np.float64(3311.3560466766357)

## 3. Footpath Zone Filtering

In [7]:
print("\n" + "="*70)
print("Footpath Zone Filtering")
print("="*70)

footpath_zones = get_footpath_zones()
zones_df = pd.DataFrame(footpath_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

# CRITICAL: Call attach_zones_to_objects ONCE
print(f"Attaching footpath zones to {len(df):,} rows...")
log_memory("Before footpath zones")

df = attach_zones_to_objects(df, gdf_zones, how="left", batch_size=100000)

log_memory("After footpath zones")
print(f"✓ Zones attached! Total rows: {len(df):,}")

df = apply_footpath_zone_filter(df)
df = df.drop(columns=['zone'], errors='ignore')
gc.collect()
log_memory("After footpath filter")


Footpath Zone Filtering
Attaching footpath zones to 50,321,862 rows...
[MEMORY] Before footpath zones: 4607.7 MB


Zone assignment batches: 100%|██████████| 504/504 [01:25<00:00,  5.88it/s]


[MEMORY] After footpath zones: 7684.0 MB
✓ Zones attached! Total rows: 50,321,862
[footpath zone] Removed 1537 objects
[MEMORY] After footpath filter: 7348.0 MB


7347.9765625

## 4. Crosswalk Zone Filtering

In [8]:
print("\n" + "="*70)
print("Crosswalk Zone Filtering")
print("="*70)

crosswalk_zones = get_crosswalk_zones()
zones_df = pd.DataFrame(crosswalk_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")
gdf_zones["orientation_deg"] = gdf_zones["geometry"].apply(compute_polygon_orientation)

print(f"Attaching crosswalk zones to {len(df):,} rows...")
log_memory("Before crosswalk zones")

df = attach_zones_to_objects(df, gdf_zones, how="left", batch_size=100000)

log_memory("After crosswalk zones")
print(f"✓ Zones attached! Total rows: {len(df):,}")

# Filter parallel vehicles
removed_ids_global = []
df_in_zones = df[df['zone'].notnull()].copy()

for zone_id in df_in_zones['zone'].unique():
    df_zone = df_in_zones[df_in_zones['zone'] == zone_id]
    orientation = gdf_zones[gdf_zones['id'] == zone_id]['orientation_deg'].iloc[0]
    parallel_ids, _ = filter_parallel_vehicles(df_zone, orientation, threshold=4.0)
    removed_ids_global.extend(parallel_ids)

df = df[~df['id'].isin(removed_ids_global)]
print(f"[crosswalk] Removed {len(removed_ids_global):,} parallel vehicles")

df = df.drop(columns=['zone'], errors='ignore')
gc.collect()
log_memory("After crosswalk filter")


Crosswalk Zone Filtering
Attaching crosswalk zones to 48,745,507 rows...
[MEMORY] Before crosswalk zones: 7348.1 MB


Zone assignment batches: 100%|██████████| 488/488 [01:15<00:00,  6.47it/s]


[MEMORY] After crosswalk zones: 7580.6 MB
✓ Zones attached! Total rows: 48,745,507
[crosswalk] Removed 9 parallel vehicles
[MEMORY] After crosswalk filter: 7349.1 MB


7349.11328125

## 5. Static Object Removal

In [9]:
print("\n" + "="*70)
print("Static Object Removal")
print("="*70)

df = filter_static_objects(df, 
    static_threshold=config['preprocessing']['static_filter']['min_speed'],
    static_ratio_min=0.8)

log_df_memory(df, "After static filter")


Static Object Removal


[static filter] Found 11,338 static objects
[static filter] Removed 11,338 static objects
  Before: 84,938 IDs (48,688,461 rows)
  After: 73,600 IDs (23,737,208 rows)
[DF MEMORY] After static filter: 1562.0 MB (23,737,208 rows)


np.float64(1561.9920272827148)

## 6. Lane Zone Assignment

In [10]:
print("\n" + "="*70)
print("Lane Zone Assignment")
print("="*70)

lane_zones = get_lane_zones()
df = assign_zones_to_vehicles(df, lane_zones)

print(df['zone'].value_counts())
print(f"\nVehicles in lanes: {(df['zone'] != 'unknown').sum():,}")

df_lanes = df[df['zone'] != 'unknown'].copy()
log_df_memory(df_lanes, "Lane vehicles")


Lane Zone Assignment

Assigning zones to 23,737,208 vehicles using spatial join...
Processing in batches of 100,000 rows


Zone assignment: 100%|██████████| 238/238 [00:48<00:00,  4.88it/s]



✓ Zone assignment complete!
  Vehicles in zones: 6,776,744
  Vehicles outside zones: 17,509,978
zone
unknown    17509978
E-L1        2033195
B-L2        1618442
E-L2         949969
C-L1         747053
B-L1         549514
C-L2         402332
D-L1         377916
D-L2          55599
A-L1          42724
Name: count, dtype: int64

Vehicles in lanes: 6,776,744
[DF MEMORY] Lane vehicles: 840.2 MB (6,776,744 rows)


np.float64(840.1648712158203)

## 7. M-DRAC Detection

In [11]:
print("\n" + "="*70)
print("M-DRAC Detection")
print("="*70)

# Generate base pairs first (EXACT OLD CODE WORKFLOW)
from ssm.utils import find_all_nearby_pairs, get_mdrac_pairs

print("\nGenerating nearby pairs...")
log_memory("Before pair generation")

# OLD code signature: find_all_nearby_pairs(df, config)
base_pairs = find_all_nearby_pairs(df_lanes, config)

print(f"✓ Generated {len(base_pairs):,} base pairs")
log_memory("After pair generation")

# Filter pairs for M-DRAC 
print("\nFiltering pairs for M-DRAC...")
mdrac_pairs = get_mdrac_pairs(base_pairs, config, skip_pair_generation=True)
print(f"✓ M-DRAC pairs after filtering: {len(mdrac_pairs):,}\"")

# Detect conflicts from filtered pairs
print("\nDetecting M-DRAC conflicts...")
mdrac_detector = ModifiedDRAC(config)
mdrac_conflicts = mdrac_detector.detect(mdrac_pairs, is_pairs_data=True)

print(f"\n{'='*70}")
print(f"M-DRAC Conflicts: {len(mdrac_conflicts):,}")
print(f"{'='*70}")



M-DRAC Detection

Generating nearby pairs...
[MEMORY] Before pair generation: 6863.9 MB
  Filtered vehicles: 2,399,306
  Generating pairs (max_distance=8.0m)...
  Processing 657,205 timestamps (batch_size=5,000)


  Pair generation: 100%|██████████| 132/132 [00:03<00:00, 36.98it/s]


  Applying overlap filter (buffer=0.5m)...
  ✓ Generated 193,181 nearby pairs (after overlap filter)
  ✓ Added zone information (zone1/zone2 columns)
✓ Generated 193,181 base pairs
[MEMORY] After pair generation: 6903.6 MB

Filtering pairs for M-DRAC...
 Approaching pairs: 87,837
  Zone filter (same lane): 38,582 pairs (filtered 49,255 different-lane)
  Lateral filter (<= 3.0m): 23,302 pairs (filtered 15,280 not aligned)
  ✓ Total filtered: 64,535 pairs | Remaining: 23,302 pairs
  Speed diff > 0.5: 11,692
  Final MDRAC pairs: 342
✓ M-DRAC pairs after filtering: 342"

Detecting M-DRAC conflicts...
 Approaching pairs: 342
  Zone filter (same lane): 342 pairs (filtered 0 different-lane)
  Lateral filter (<= 3.0m): 342 pairs (filtered 0 not aligned)
  ✓ Total filtered: 0 pairs | Remaining: 342 pairs
  Speed diff > 0.5: 342
  Final MDRAC pairs: 342

M-DRAC Conflicts: 3


In [12]:
# Save M-DRAC results
if len(mdrac_conflicts) > 0:
    mdrac_path = save_detection_results(mdrac_conflicts, OUTPUT_DIR, 'mdrac', 'brussels', START_DATE)
    print(f"Saved to {mdrac_path}")

✓ Saved 3 conflicts to /home/ubuntu/prem/results/brussels/mdrac/05/mdrac_05.csv
Saved to /home/ubuntu/prem/results/brussels/mdrac/05/mdrac_05.csv


## Complete

In [13]:
print("\n" + "="*70)
print("BRUSSELS ANALYSIS COMPLETE")
print("="*70)
print(f"M-DRAC: {len(mdrac_conflicts):,}")
print("="*70)


BRUSSELS ANALYSIS COMPLETE
M-DRAC: 3
